# feral — indefinite KKT systems & certified inertia

Interior-point and equality-constrained problems produce **symmetric indefinite** saddle-point (KKT) systems

$$ K = \begin{bmatrix} H & A^\top \\ A & 0 \end{bmatrix} $$

with `H` (n×n) the Hessian and `A` (m×n) the constraint Jacobian. For a well-posed problem with `H` positive definite on the null space of `A`, the KKT matrix has inertia `(n, m, 0)`: `n` positive, `m` negative, `0` zero eigenvalues. `feral` reports this exactly, which is how an IPM checks it is on a descent step.

In [1]:
import numpy as np
import scipy.sparse as sp
import feral

rng = np.random.default_rng(1)

## Build a small KKT matrix

In [2]:
n, m = 8, 3
# SPD Hessian H = M M^T + I
M = rng.standard_normal((n, n))
H = M @ M.T + np.eye(n)
# full-rank constraint Jacobian A (m x n)
Acon = rng.standard_normal((m, n))

K = np.block([
    [H,            Acon.T],
    [Acon,         np.zeros((m, m))],
])
K_sp = sp.csc_matrix(K)
print('KKT dim =', K.shape[0], '= n + m =', n + m)

KKT dim = 11 = n + m = 11


## Factor and verify the inertia

We expect `(n, m, 0)`. You can also pass `expected_inertia` to `factor`; it returns `WRONG_INERTIA` (without invalidating the factor) if the count differs — the signal an IPM uses to perturb.

In [3]:
K_feral = feral.from_scipy(K_sp, symmetric='full')
solver = feral.Solver()
expected = feral.Inertia(n, m, 0)
status, inertia = solver.factor(K_feral, expected_inertia=expected)
print('status  :', feral.FactorStatus(status).name)
print('inertia :', inertia)
print('expected:', expected)
assert inertia == expected

status  : SUCCESS
inertia : Inertia(n_pos=8, n_neg=3, n_zero=0)
expected: Inertia(n_pos=8, n_neg=3, n_zero=0)


## Cross-check the inertia against a dense eigendecomposition

In [4]:
eig = np.linalg.eigvalsh(K)
n_pos = int(np.sum(eig > 1e-9))
n_neg = int(np.sum(eig < -1e-9))
n_zero = int(np.sum(np.abs(eig) <= 1e-9))
print(f'eig-based inertia = ({n_pos}, {n_neg}, {n_zero})')
assert (n_pos, n_neg, n_zero) == inertia.as_tuple()

eig-based inertia = (8, 3, 0)


## Solve the KKT system and check the residual

In [5]:
b = rng.standard_normal(n + m)
x = solver.solve_refined(K_feral, b)
res = np.max(np.abs(K @ x - b))
print(f'max abs residual = {res:.3e}')
assert res < 1e-9

max abs residual = 6.939e-16


## A genuinely singular case

If `A` is rank-deficient the KKT matrix is singular; the inertia then carries a non-zero `n_zero`. feral reports it rather than silently returning garbage.

In [6]:
Acon_rd = Acon.copy()
Acon_rd[-1] = Acon_rd[0]          # duplicate a row -> rank deficient
K_rd = np.block([[H, Acon_rd.T], [Acon_rd, np.zeros((m, m))]])
eig_rd = np.linalg.eigvalsh(K_rd)
print('dense zero eigenvalues:', int(np.sum(np.abs(eig_rd) <= 1e-8)))
print('(feral flags singular / non-zero n_zero on such systems)')

dense zero eigenvalues: 1
(feral flags singular / non-zero n_zero on such systems)
